# **Damped Burgers** — $-\tfrac{\lambda}{2}\partial_x(\phi^2)$ (composite $\partial_x$ vertex, d = 1)

$$\partial_t\phi = -\mu\phi + D\,\partial_x^2\phi - \tfrac{\lambda}{2}\partial_x(\phi^2) + \eta
  \;=\; -\mu\phi + D\,\partial_x^2\phi - \lambda\,\phi\,\partial_x\phi + \eta,\qquad
  \langle\eta\eta\rangle = 2T\,\delta\,\delta.$$

The linear damping $-\mu\phi$ gives a massive, IR-safe propagator and fixes the homogeneous saddle
$\phi^*=0$.  The $\partial_x$ acts on the $\phi^2$ **composite** — a *first*-derivative vertex, so its
form factor is **imaginary** ($\partial_x\to ik$), unlike Model B's real $\nabla^2\to-k^2$.
Expanding about the saddle also makes a bilinear $\propto\phi^*\,\partial_x(\delta\phi)$, a propagator
**drift** that vanishes at $\phi^*=0$ (the drift-generalized heat kernel handles it).

In [ ]:
import os, sys, time
# --- depth-robust repo root: walk up until we find the 'pipeline' package ---
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'pipeline')):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
os.chdir(os.path.join(_root, 'notebooks'))  # cwd=notebooks/ so relative paths resolve
import numpy as np
import matplotlib.pyplot as plt
from pipeline.compute import compute_cumulants
from pipeline.theory import TheoryBuilder
from models.spatial_field_1d_sim import simulate, equal_time_correlator

def order_label(ell):
    """0->'tree', 1->'tree + 1-loop', 2->'tree + 1-loop + 2-loop'."""
    return ('tree' if ell == 0 else
            'tree + ' + ' + '.join('%d-loop' % j for j in range(1, ell + 1)))

mu, D, lam, T = 1.0, 1.0, 0.3, 1.0       # mass, diffusion, Burgers coupling, noise temp

def build_theory():
    return (TheoryBuilder('burgers-1d', n_populations=0)
            .physical_field('phi', spatial_dim=1)
            .parameter('mu',  default=mu,  domain='positive')
            .parameter('D',   default=D,   domain='positive')
            .parameter('lam', default=lam, domain='real')
            .parameter('T',   default=T,   domain='positive')
            .equation(lhs='(Dt + mu - D*Laplacian)*phi', rhs='0')   # gradient forcing vanishes at φ*=0
            .set_action_text('phit*(Dt(phi) + mu*phi - D*Lap(phi) + (lam/2)*Dx(phi^2, 0)) - T*phit^2')
            .operator_ir()
            .boundary('infinite').initial('stationary').build())

## 0. Choose the order — `k` and `ℓ`

In [ ]:
# ============================  CHOOSE THE ORDER  ============================
MAX_ELL    = 1      # loop order ℓ:  0 = tree,  1 = +1-loop  (ℓ=2 is correct but slow)
K_EXTERNAL = 2      # correlator order k:  2 = two-point ⟨phiphi⟩
VERBOSE    = True   # True ⇒ print the staged [1/7]…[7/7] pipeline for each order
# ===========================================================================

if K_EXTERNAL != 2:
    raise NotImplementedError("v1 implements the k=2 two-point correlator.")

xs = np.linspace(0.0, 6.0, 25)                       # output separations x ≥ 0
kw = dict(k=K_EXTERNAL, external_fields=[('phi', 1), ('phi', 1)], spatial_grid=xs,
          tau_max=0.0, tau_step=1.0,                 # equal-time only (τ=0) — fast
          verbose=VERBOSE, use_cache=False, mf_dae_n_starts=4)
fund = {'mu': mu, 'D': D, 'lam': lam, 'T': T}
orders = list(range(0, MAX_ELL + 1))
print('will compute orders ℓ =', orders, ' at k =', K_EXTERNAL)

## 1. Theory — every order up to `MAX_ELL` through `compute_cumulants`

`Dx(phi^2, 0)` lowers to a **composite first-derivative vertex** (`mode='composite'`, $\partial_x\to ik$,
imaginary).  The pipeline takes the real part at the output (`imag_frac=0` — the physical correlator is
real).  The bilinear $\partial_x$ cross-term becomes a propagator drift $V\propto\phi^*$ that is $0$ at
the saddle, so the propagator reduces to the pure heat kernel.

In [ ]:
curves = {}
for ell in orders:
    t0 = time.time()
    out = compute_cumulants(build_theory(), max_ell=ell, fundamental=fund, **kw)
    mid = out['C_tau_x'].shape[0] // 2               # τ = 0 row
    curves[ell] = np.real(out['C_tau_x'])[mid]
    si = out.get('spatial_info', {}) or {}
    print('%-26s C(0,0) = %.4f   mode = %s   live diagrams = %s   (%.0fs)'
          % (order_label(ell), curves[ell][0], si.get('vertex_mode', '—'),
             si.get('n_live_diagrams', '—'), time.time() - t0))

print('\nvariance C(0,0) by cumulative order:')
for ell in orders:
    print('   %-26s = %.4f' % (order_label(ell), curves[ell][0]))

## 2. Simulation of the SPDE

The Burgers nonlinearity enters the spectral integrator as `−(λ/2)·ik·rfft(φ²)` (the `lam_burg`
forcing; its $k=0$ component vanishes, so Burgers is conservative — no excess velocity).

In [ ]:
snaps, x_grid, meta = simulate(L=40.0, N=256, mu=mu, D=D, T=T, lam_burg=lam,
                               dt=None, n_steps=120000, burn_in=20000,
                               record_every=20, seed=1)
if not np.all(np.isfinite(snaps)) or np.max(np.abs(snaps)) > 30:
    print('WARNING: the simulation blew up (gradient/conserved stiffness) — '
          'reduce the coupling, raise N, or shrink dt.')
mean = float(np.mean(snaps))                         # ⟨φ⟩ (excess velocity for KPZ; ~0 otherwise)
Cx_full = equal_time_correlator(snaps) - mean**2     # CONNECTED correlator (subtract ⟨φ⟩²)
half = len(x_grid) // 2 + 1
xc, Cx = x_grid[:half], Cx_full[:half]               # one period side, x ≥ 0
print('sim ⟨φ⟩ = %.4f   sim connected C(0,0) = %.4f' % (mean, Cx[0]))
print('theory C(0,0) by cumulative order:')
for ell in orders:
    print('   %-26s = %.4f' % (order_label(ell), curves[ell][0]))

## 3. Compare — theory orders vs simulation

Burgers is conservative (the $k=0$ forcing vanishes), so the equal-time variance shift is small and
negative — a weak but sign-correct end-to-end target.

In [ ]:
styles = {0: ('--', 'C0', r'tree  $C_0(x)$'),
          1: ('-',  'C3', 'tree + 1-loop'),
          2: ('-',  'C2', 'tree + 2-loop')}

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for a in ax:
    for ell in orders:
        ls, col, lab = styles[ell]
        a.plot(xs, curves[ell], ls, lw=2, color=col, label=lab)
    a.plot(xc, Cx, 'o', ms=5, color='k', alpha=0.7, label='simulation (connected)')
    a.set_xlabel('x'); a.set_ylabel('C(x, 0)'); a.set_xlim(0, xs.max())
ax[0].set_title('equal-time correlator $C(x,0)$'); ax[0].legend()
ax[1].set_yscale('log'); ax[1].set_title('log scale'); ax[1].legend()
plt.tight_layout(); plt.show()

sim0 = Cx[0]
print('distance |sim − theory| at x=0  (sim connected variance = %.4f):' % sim0)
for ell in orders:
    print('   %-26s : |Δ| = %.4f' % (order_label(ell), abs(sim0 - curves[ell][0])))

## Summary

Burgers is the **composite first-derivative** representative ($\partial_x$ on the $\phi^2$ composite,
imaginary form factor).  The same machinery as Model B, with $\partial_x\to ik$ carried complex and the
saddle-drift handled by the drift-generalized heat kernel.

**Knobs:** `MAX_ELL`, `lam` (Burgers coupling, the loop shift $\propto\lambda^2$), `mu`, `D`, `T`.
The connected variance shift is small and negative (sim $-0.0002\pm0.0002$ vs theory $-0.0001$ at
$\lambda=0.3$ — `docs/kpz_burgers_sim_validation.py`).